<a href="https://colab.research.google.com/github/FelixvanDijk/DL-Lab5/blob/main/lab-5-transformers/lab-5-transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5: Training a Transformer

In today's lab session we will:

1. [Build a Simple Transformer Model](#Build-a-Simple-Transformer-Model)
2. [Expand and Compare to a More Complex Transformer Model](#Expanding-to-a-More-Complex-Transformer-Model)
3. [Adding a Classification Token (CLS)](#Adding-a-Classification-Token)
4. [Adding Positional Embeddings](#Adding-Positional-Embeddings)
5. [Extension - Multihead Attention from scratch (mostly)](#Multihead-Attention-from-scratch)

As a note, this lab is more hands off than previous labs - the transformer code is _relatively_ simple due to the in-built functions in PyTorch so most of this labsheet (and lab) is directed at your ability to train a model and compare findings. You can use all that you have learnt over the past 4 labs to apply here - think of this lab as the final boss of the formative Appied Deep Learning exercises.

As always, if you get stuck or are unsure about something, please ask the lecturers and/or the teaching assistants.

In [ ]:
#!/usr/bin/env python3
import time
from multiprocessing import cpu_count
from typing import Union, NamedTuple

import torch
import torch.backends.cudnn
import numpy as np
from torch import nn, optim
from torch.nn import functional as F
import torchvision.datasets
from torch.optim.optimizer import Optimizer
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms

import argparse
from pathlib import Path

torch.backends.cudnn.benchmark = True

parser = argparse.ArgumentParser(
    description="Train a simple CNN on CIFAR-10",
    formatter_class=argparse.ArgumentDefaultsHelpFormatter,
)
default_dataset_dir = Path.home() / ".cache" / "torch" / "datasets"
parser.add_argument("--dataset-root", default=default_dataset_dir)
parser.add_argument("--log-dir", default=Path("logs"), type=Path)
parser.add_argument("--learning-rate", default=1e-1, type=float, help="Learning rate")
parser.add_argument(
    "--batch-size",
    default=128,
    type=int,
    help="Number of images within each mini-batch",
)
parser.add_argument(
    "--epochs",
    default=20,
    type=int,
    help="Number of epochs (passes through the entire dataset) to train for",
)
parser.add_argument(
    "--val-frequency",
    default=2,
    type=int,
    help="How frequently to test the model on the validation set in number of epochs",
)
parser.add_argument(
    "--log-frequency",
    default=10,
    type=int,
    help="How frequently to save logs to tensorboard in number of steps",
)
parser.add_argument(
    "--print-frequency",
    default=10,
    type=int,
    help="How frequently to print progress to the command line in number of steps",
)
parser.add_argument(
    "-j",
    "--worker-count",
    default=cpu_count(),
    type=int,
    help="Number of worker processes used to load data.",
)
parser.add_argument("--data-aug-hflip", action="store_true", help="Apply horizontal flip data augmentation")
parser.add_argument("--data-aug-crop", action="store_true", help="Apply random crop data augmentation")
parser.add_argument("--data-aug-brightness", default=0.0, type=float, help="Amount to jitter brightness")
parser.add_argument("--dropout", default=0.0, type=float, help="Probability of dropping out an input")

class ImageShape(NamedTuple):
    height: int
    width: int
    channels: int

class CNN(nn.Module):
    def __init__(
        self,
        height: int = 32,
        width: int = 32,
        channels: int = 3,
        class_count: int = 10,
        dropout: float = 0.0
    ):
        super().__init__()
        self.input_shape = ImageShape(height, width, channels)
        self.class_count = class_count

        self.conv1 = nn.Conv2d(
            in_channels=self.input_shape.channels,
            out_channels=32,
            kernel_size=(5, 5),
            padding=(2, 2),
        )
        self.initialise_layer(self.conv1)

        self.bn1 = nn.BatchNorm2d(num_features=32)
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=(5, 5),
            padding=(2, 2),
        )
        self.initialise_layer(self.conv2)

        self.bn2 = nn.BatchNorm2d(num_features=64)
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))

        self.fc1 = nn.Linear(4096, 1024)
        self.initialise_layer(self.fc1)

        self.bn_fc1 = nn.BatchNorm1d(num_features=1024)
        self.dropout = nn.Dropout(p=dropout)

        self.fc2 = nn.Linear(1024, self.class_count)
        self.initialise_layer(self.fc2)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.conv1(images)))
        x = self.pool1(x)

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)

        x = torch.flatten(x, start_dim=1)

        # Apply dropout to the input of FC1
        x = self.dropout(x)
        x = F.relu(self.bn_fc1(self.fc1(x)))

        # Apply dropout to the input of FC2
        x = self.dropout(x)
        x = self.fc2(x)

        return x

    @staticmethod
    def initialise_layer(layer):
        if hasattr(layer, "bias") and layer.bias is not None:
            nn.init.zeros_(layer.bias)
        if hasattr(layer, "weight") and layer.weight is not None:
            nn.init.kaiming_normal_(layer.weight)

class Trainer:
    def __init__(
        self,
        model: nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        criterion: nn.Module,
        optimizer: Optimizer,
        summary_writer: SummaryWriter,
        device: torch.device
    ):
        self.model = model.to(device)
        self.device = device
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.summary_writer = summary_writer
        self.step = 0

    def train(
        self,
        epochs: int,
        val_frequency: int,
        print_frequency: int,
        log_frequency: int
    ):
        self.model.train()
        for epoch in range(epochs):
            self.model.train()
            data_load_start_time = time.time()
            for batch, labels in self.train_loader:
                batch = batch.to(self.device)
                labels = labels.to(self.device)
                data_load_time = time.time() - data_load_start_time
                step_start_time = time.time()

                logits = self.model.forward(batch)
                loss = self.criterion(logits, labels)

                loss.backward()

                self.optimizer.step()
                self.optimizer.zero_grad()

                self.step += 1
                step_time = time.time() - step_start_time

                if self.step % print_frequency == 0:
                    try:
                        preds = logits.argmax(dim=-1)
                        accuracy = compute_accuracy(labels, preds)
                    except NameError:
                        accuracy = 0.0

                    print(
                        f"epoch: [{epoch}], step: [{self.step}], batch loss: {loss.item():.5f}, "
                        f"batch accuracy: {accuracy * 100:2.2f}, data load time: {data_load_time:.5f}, "
                        f"step time: {step_time:.5f}"
                    )

                if self.step % log_frequency == 0:
                    self.summary_writer.add_scalar("loss/train", loss.item(), self.step)
                    try:
                        self.summary_writer.add_scalar("accuracy/train", accuracy, self.step)
                    except NameError:
                        pass

                data_load_start_time = time.time()

            if (epoch + 1) % val_frequency == 0:
                self.validate()

    def validate(self):
        results = {"preds": [], "labels": []}
        total_loss = 0
        self.model.eval()

        with torch.no_grad():
            for batch, labels in self.val_loader:
                batch = batch.to(self.device)
                labels = labels.to(self.device)
                logits = self.model(batch)
                loss = self.criterion(logits, labels)
                total_loss += loss.item()
                preds = logits.argmax(dim=-1).cpu().numpy()
                results["preds"].extend(list(preds))
                results["labels"].extend(list(labels.cpu().numpy()))

        preds_np = np.array(results["preds"])
        labels_np = np.array(results["labels"])
        accuracy = compute_accuracy(labels_np, preds_np)
        average_loss = total_loss / len(self.val_loader)

        self.summary_writer.add_scalars("accuracy", {"test": accuracy}, self.step)
        self.summary_writer.add_scalars("loss", {"test": average_loss}, self.step)
        print(f"validation loss: {average_loss:.5f}, overall accuracy: {accuracy * 100:2.2f}")

        # --- PER-CLASS ACCURACY TASK ---
        classes = [
            "airplane", "automobile", "bird", "cat", "deer",
            "dog", "frog", "horse", "ship", "truck"
        ]
        print("\nPer-Class Accuracy Breakdown:")
        for i, class_name in enumerate(classes):
            class_mask = (labels_np == i)
            total_class_samples = class_mask.sum()
            if total_class_samples > 0:
                class_correct = ((preds_np == i) & class_mask).sum()
                class_accuracy = float(class_correct) / total_class_samples
                print(f"  {class_name:<11}: {class_accuracy * 100:6.2f}%")
                self.summary_writer.add_scalar(f"accuracy_class/{class_name}", class_accuracy, self.step)
            else:
                print(f"  {class_name:<11}: No samples found in validation batch.")
        print()

def compute_accuracy(
    labels: Union[torch.Tensor, np.ndarray], preds: Union[torch.Tensor, np.ndarray]
) -> float:
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()
    if isinstance(preds, torch.Tensor):
        preds = preds.cpu().numpy()
    assert len(labels) == len(preds)
    return float((labels == preds).sum()) / len(labels)

def get_summary_writer_log_dir(args: argparse.Namespace) -> str:
    tb_log_dir_prefix = (
        f"CNN_bn_"
        f"dropout={args.dropout}_"
        f"bs={args.batch_size}_"
        f"lr={args.learning_rate}_"
        f"momentum=0.9_"
        f"brightness={args.data_aug_brightness}_" +
        ("hflip_" if args.data_aug_hflip else "") +
        ("crop_" if args.data_aug_crop else "") +
        f"run_"
    )
    i = 0
    while i < 1000:
        tb_log_dir = args.log_dir / (tb_log_dir_prefix + str(i))
        if not tb_log_dir.exists():
            return str(tb_log_dir)
        i += 1
    return str(tb_log_dir)

def main(args):
    if torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
        print("CUDA not available, running on CPU instead.")

    # Test pipeline (unaltered evaluation transforms)
    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    # Training pipeline with conditional augmentations
    train_transform_list = []
    if args.data_aug_hflip:
        train_transform_list.append(transforms.RandomHorizontalFlip())
    if args.data_aug_crop:
        train_transform_list.append(transforms.RandomCrop(32, padding=4))
    if args.data_aug_brightness > 0:
        train_transform_list.append(transforms.ColorJitter(brightness=args.data_aug_brightness))
    train_transform_list.extend([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    train_transform = transforms.Compose(train_transform_list)

    train_dataset = torchvision.datasets.CIFAR10(
        root=args.dataset_root, train=True, download=True, transform=train_transform
    )
    val_dataset = torchvision.datasets.CIFAR10(
        root=args.dataset_root, train=False, download=True, transform=val_transform
    )

    train_loader = DataLoader(
        train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=args.worker_count
    )
    val_loader = DataLoader(
        val_dataset, batch_size=args.batch_size, shuffle=False, num_workers=args.worker_count
    )

    model = CNN(dropout=args.dropout)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=args.learning_rate, momentum=0.9)

    log_dir = get_summary_writer_log_dir(args)
    summary_writer = SummaryWriter(log_dir=log_dir)

    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        summary_writer=summary_writer,
        device=device,
    )

    trainer.train(
        epochs=args.epochs,
        val_frequency=args.val_frequency,
        print_frequency=args.print_frequency,
        log_frequency=args.log_frequency,
    )

    summary_writer.close()

if __name__ == "__main__":
    import shutil
    from google.colab import drive
    from torchvision.datasets.utils import check_integrity, extract_archive

    drive.mount("/content/drive")
    args = parser.parse_args([])

    saved_archive = Path(
        "/content/drive/MyDrive/deep_learning_lab/datasets/"
        "cifar-10-python.tar.gz"
    )

    args.dataset_root = Path("/content/cifar10_data")
    args.dataset_root.mkdir(parents=True, exist_ok=True)

    dataset_class = torchvision.datasets.CIFAR10
    dataset_folder = args.dataset_root / dataset_class.base_folder

    required_files = (
        dataset_class.train_list
        + dataset_class.test_list
        + [(dataset_class.meta["filename"], dataset_class.meta["md5"])]
    )

    def local_dataset_is_valid():
        return all(
            check_integrity(str(dataset_folder / filename), checksum)
            for filename, checksum in required_files
        )

    if not local_dataset_is_valid():
        if not check_integrity(str(saved_archive), dataset_class.tgz_md5):
            raise RuntimeError(
                "The saved CIFAR-10 archive is missing or corrupted. "
                "Check My Drive/deep_learning_lab/datasets/"
                "cifar-10-python.tar.gz. No download was attempted."
            )

        local_archive = args.dataset_root / dataset_class.filename

        print("Copying saved CIFAR-10 from Google Drive...")
        shutil.copy2(saved_archive, local_archive)

        if not check_integrity(str(local_archive), dataset_class.tgz_md5):
            raise RuntimeError("The local archive copy failed verification.")

        print("Extracting locally...")
        extract_archive(str(local_archive), str(args.dataset_root))

        if not local_dataset_is_valid():
            raise RuntimeError("Extracted dataset failed verification.")

    print("Using verified CIFAR-10 files — no internet download needed.")

    args.log_dir = Path("/content/drive/MyDrive/deep_learning_lab/logs")
    args.log_dir.mkdir(parents=True, exist_ok=True)
    args.worker_count = 2

    main(args)

---
# Build a Simple Transformer Model

For the whole lab, we will focus on recreating a simpler version of the VIT architecture which can be seen below:

<center><img alt="VIT Transformer Diagram" src="https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-5-transformers/media/vit_ex.png?raw=1" style="max-height: 300px;"></center>

Initially we'll be building the simplest Transformer network of a single encoder layer and a single FC layer for classification.

From the figure, you can see that the image is split into patches, which are small regions of the image that are fed through the transformer model. This allows for relational modelling between different parts of the image.

The first step is to implement this 'patchifying process'...

## Patchifying an Image

The input shape for our images are of size $C \times H \times W$, during training these will be represented by a 4-$D$ tensor which also includes the batch size ($B \times C \times H \times W$).

We want to break our image into $n \times n$ patches. **Note** that $n$ should divide evenly into $H$ and $W$.

**TASK 1:** Implement the patchifying function to convert your input tensor into the form of $B \times ((H \times W)/n^2) \times (n^2 \times C)$. **Note** that this is 3-dimensional.
Look at [`nn.Unfold`](https://pytorch.org/docs/stable/generated/torch.nn.Unfold.html) to do this simply. This operation groups together inputs across a specific kernel size for 4-$D$ inputs. *Hint:* kernel size is your $n$, think back to the definition of pooling layers for your stride value.

Check that your patchify function works with $n=8$, which we will be using for most of this labsheet.

In [1]:
import torch
import torch.nn as nn

def patchify(images, n):
    """
    Converts a batch of images of shape (B, C, H, W) into patches of shape
    (B, (H * W) / n^2, n^2 * C).
    """
    B, C, H, W = images.shape

    # nn.Unfold extracts patches of size n x n with stride n (no overlap)
    # Output of unfold is shape: (B, C * n * n, L) where L is the number of patches: (H*W)/n^2
    unfold = nn.Unfold(kernel_size=n, stride=n)
    patches = unfold(images)

    # We need to reshape/transpose to: (B, L, n^2 * C)
    # First swap the dimensions of patches to (B, L, C * n * n)
    patches = patches.transpose(1, 2)

    return patches

# --- Verification Check ---
# Let's create a dummy batch of CIFAR-like images: 128 images of 3 channels, 32x32 pixels
B, C, H, W = 128, 3, 32, 32
n = 8
dummy_images = torch.randn(B, C, H, W)

output_patches = patchify(dummy_images, n)
print(f"Input shape: {dummy_images.shape}")
print(f"Expected output shape: ({B}, {int((H*W)/(n**2))}, {int((n**2)*C)})")
print(f"Actual output shape:   {output_patches.shape}")

Input shape: torch.Size([128, 3, 32, 32])
Expected output shape: (128, 16, 192)
Actual output shape:   torch.Size([128, 16, 192])


## A Simple Transformer Encoder

For now we are going to skip the linear projection of flattened patches - generally this is to reduce the embedding size so the fully connected layers within the transformer don't grow too big. Instead we're going to start looking at the transformer encoder. Specifically, we will be creating a model with one Transformer Encoder Layer and one FC layer for the classification (the MLP head in the image). An example of the first transformer model can be seen below:

<center><img alt="Step 1 Transformer Diagram" src="https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-5-transformers/media/step_1.png?raw=1" style="max-height: 500px;"></center>

Recall from the lectures that Transformers have both an encoder and a decoder section, but we are doing classification instead of reconstruction so we do not need a decoder. With this in mind lets look at the PyTorch implementations, we'll highlight three different structures that are available:

* [`nn.Transformer`](https://pytorch.org/docs/stable/generated/torch.nn.Transformer.html#torch.nn.Transformer): This allows you to create an entire Transformer model with both encoders and decoders (each having differing numbers of encoder/decoder layers). As we are focusing only on an encoder we will skip this.
* [`nn.TransformerEncoder`](https://pytorch.org/docs/stable/generated/torch.nn.TransformerEncoder.html): This allows you to create only the encoder portion of the transformer which can consist of multiple encoder layers.
* [`nn.TransformerEncoderLayer`](https://pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html): This creates only a single encoder layer in the transformer - the smallest atomic unit. This still contains both the Multi-Head Self Attention and the MLP layers which collectively make up *one layer* as well as the activation functions and layer norms (you can think of this as a much higher level than previous labs where we have to define all of these functions!)



The `TransformerEncoderLayer` has a number of arguments, but we only need to worry about three for now: `d_model`, `n_heads`, and `batch_first`. The first specifies the embedding size of the input and the internal layers; the second gives the number of heads in the model; and the third specifies whether the input is (Batch, Sequence, Feature) (Sequence, Batch, Feature).

* As we're starting simple, set `n_heads=1` so that we only have a single head in our self-attention layer - we can explore how the number of heads change the performance later.
* The value of `d_model` needs to be the size of your patches ($n^2 \times C=192$).
* As the dataset returns the data in the form (Batch, Sequence, Feature) we need to set `batch_first` to `True` (this is `False` by default!)

**TASK 2**: Add the transformer encoder layer in the constructor for the `CIFAR_Transformer` class and add it to the forward method. Remember to patchify your input using the result of task 1 before passing this into your transformer.

Run your code as a test, printing out the shape of the output of the model after one batch - your program won't fully run yet but this is fine!

**Assessed Question:** What is the expected output shape from the transformer?


In [3]:
import torch
import torch.nn as nn

class CIFAR_Transformer(nn.Module):
    def __init__(self, n=8):
        super().__init__()
        self.n = n
        self.d_model = (n**2) * 3  # For n=8 and C=3, this is 192
        self.num_classes = 10

        # TASK 2: Transformer Encoder Layer
        # d_model=192, nhead=1, batch_first=True
        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.d_model,
            nhead=1,
            batch_first=True
        )

        # TASK 3.1: Add fully connected layer for classification
        self.fc = nn.Linear(self.d_model, self.num_classes)

    def forward(self, x):
        # Patchify the input using the function from Task 1
        patches = patchify(x, self.n)

        # Pass through the transformer encoder
        out = self.encoder_layer(patches)

        # TASK 3.2: Global average pooling across the sequence dimension
        out = out.mean(dim=1)

        # Pass through the fully connected classification head
        logits = self.fc(out)

        return logits

# --- Verification Check ---
model = CIFAR_Transformer(n=8)
dummy_input = torch.randn(128, 3, 32, 32) # Batch of 128

output = model(dummy_input)
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape from transformer: {output.shape}")
assert output.shape == (128, 10), "Output shape is incorrect!"

Input shape: torch.Size([128, 3, 32, 32])
Output shape from transformer: torch.Size([128, 10])


**Assessed Question Answer:**

The expected output shape from the transformer is `(128, 16, 192)` (or more generally `(B, (H*W)/n^2, n^2*C)`).

This is because the `TransformerEncoderLayer` preserves both the sequence length (number of patches, 16) and the embedding dimension (patch size flattened, 192) of the input. It performs a sequence-to-sequence mapping without changing the dimensionalities.

With this implemented we can start to think of the output and how to turn this into the logits we need for classification. After the output of the transformer our output is too large to pass into a fully connected layer. Most methods use a specific input token which we'll explore later, but for now we are going to average across all outputs (known as global average pooling) for an example and then pass this into the MLP head.

**TASK 3.1:** Add in a fully connected layer into the constructor of `CIFAR_Transformer` which projects from the dimension of your transformer (192) to the number of classes (10).

**TASK 3.2:** Average using mean the output of the Transformer Encoder before passing this into the fully connected layer - pay attention to which dimension to average over! After this, the forward method should return logits that can be used for classification.

<details>
  <summary>Hint</summary>
  
  You will need to average along the middle dimension of the output of the transformer encoder layer, then this can be passed straight into the fully connected layers. Look at Lab 1 or Lab 2 code if you need a refresher on how to do this.
</details>

Run your code and check the output of the forward pass is of size $B \times 10$ where $B$ is your batch size (128 by default).

**TASK 4:** Remove the `sys.exit()` command in the training loop and now you can fully train your transformer model.


The final thing we'll look at before creating a more complex transformer is looking at the number of parameters within the model. We can do this with the following code:

```python
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
```

**TASK 5:** Add count_parameters into your `train_cifar.py` script at the start of your traininer and then compare the number of parameters in your transformer model to the CNN model that we trained in Lab 2.

How does your accuracy compare with the CNN model with the differing number of parameters? Discuss in pairs and/or with a Lecturer/Teaching Assistant as to your findings.

**Assessed Question:** Although increasing the number of parameters can enhance performance, what is a potential downside?

---

# Expanding to a More Complex Transformer Model

We will now expand to a more complex transformer model, namely by adding in extra heads to the MHSA (as until now we have only been using Self Attention!), multiple encoder layers and an MLP for the classification head. The goal for this part is to create the transformer below:

<center><img alt="Step 2 Transformer Diagram" src="https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-5-transformers/media/step_2.png?raw=1" style="max-height: 500px;"></center>

## Increasing the Number of Heads

The first and easiest thing to modify is to increase the number of heads within the self attention of the transformer model. You may have noticed one of the input arguments to the script is `args.num_heads`.

**TASK 6:** Run your program comparing using differing numbers of heads. How does increasing the number of heads effect the performance of the model?
<details>
  <summary>Hint</summary>

  If you want an idea of what heads to run, it is normal to ablate in powers of 2, so try 1, 2, 4, 8 as a starting point.
</details>

**Assessed Question:**: In the [`Attention Is All You Need`](https://arxiv.org/abs/1706.03762) paper, which popularised the modern transformer architecture, how are the outputs from multiple heads combined into a final value?

## Increasing the Size of the MLP Head

As of now we have used only a single FC layer to project the output of the transformer encoder to the logits for classification, we will now add an extra layer to allow a more complex projection and *hopefully* increase performance!

**TASK 7:** Add an extra fully connected layer to the classification head in both the constructor and the forward method. We suggest a hidden layer size of 100. In pairs and/or with a teacher discuss how this effects performance and number of parameters in the model.

## Increasing the Number of Encoder Layers

Another aspect of increasing the complexity of the model is to increase the number of encoder layers within the model. It is common nowadays for transformer models to have 6 or 8 layers within them, but due to CIFAR-10's complexity, we'll only compare with adding an extra 1 or 2 layers.

**TASK 8:** Exploring adding an extra 1 or 2 encoder layers into the constructor and forward method to see how it effects the performance and number of parameters.

<details>
  <summary>Hint</summary>

Look up [`nn.modulelist`](https://pytorch.org/docs/stable/generated/torch.nn.ModuleList.html) which can allow for a variable number of modules within a Deep Learning model and save you hard-coding specific numbers of layers.
</details>

## Ablation Studies

We now have a large number of different model setups and types. From our base model we can use: $[1, 2, 4, 8]$ heads; $[1, 2]$ fully connected layers; and $[1, 2, 3]$ encoder layers. This doesn't even include different size patches, we could also try using $n=4$ or $n=16$ to see how this effects performance (we recommend finishing the rest of this labsheet before running these experiments though!)

If we want to perform a grid search, i.e. searching exhaustively through all options, this explodes exponentially with every new setup, we would have to run $4 \times 2 \times 3 \times 3 = 72$ different experiments!

Instead, to find the best performance it is common to build this iteratively using best case performance. For example, we run $4$ experiments to find the best number of heads, then fix that value and then run $2$ experiments for the fully connected layers and fix that, etc.

In this way, we reduce the number of experiments from $72$ to $12$ which is a lot, but much more manageable.

---
# Adding a Classification Token


If you look at the VIT transformer diagram again, you will notice that a classification (CLS) token is passed into the model and only this output is then utilised in the classification head to predict the class of the image. This is a common trick as the learnt parameters of the CLS token can be fine-tuned alongside the weights of the model. Combined with the multi-head self-attention layers, the classification token will be influenced by all other inputs and can be thought of as a summary.

<center><img alt="VIT Transformer Diagram" src="https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-5-transformers/media/vit_ex.png?raw=1" style="max-height: 300px;"></center>

In this section, we will look at implementing the CLS token and training the best performing transformer model with the CLS token.

**TASK 9.1:** Define a CLS token within the constructor as an [`nn.Parameter`](https://pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html). You can do this by modifying the code below which creates a learnable parameter in your model:

```
nn.Parameter(torch.randn(<args>))
```

**Assessed Question:** What does the nn.Parameter function allow us to do?

**TASK 9.2:** In the forward method of your model, prepend your CLS token to the input features *after* they have been turned into patches. **Note:** you will need to repeat your CLS token so that the dimensions match, *i.e.* your CLS token is of size $(1, 1, D)$ and your inputs are of size $(B, 16, D)$ and you want a result of $(B, 17, D)$. You can use [`torch.Tensor.Expand`](https://pytorch.org/docs/stable/generated/torch.Tensor.expand.html) and [`torch.cat`](https://pytorch.org/docs/stable/generated/torch.cat.html) to do this.

Afterwards, run your model to see how performance/number of parameters compares to without using the CLS token.

## Removing Global Pooling

Paying closer attention to the VIT transformer diagram above, we can see that only the output from the CLS token is used in the classification head. We can do the same by indexing the output of final encoder layer using `x[:,0,:]` or similar. **NOTE:** make sure you choose the output that matches the CLS token!

**TASK 10:** In the forward method of your method, use only the output corresponding to the CLS token as input to the classification head.

Run your model to check its performance and compare to the global pooling above.

---
# Adding Positional Embeddings

The final part that's missing from our implementation and the VIT model are the positional embeddings. Recall from the lecture, these are added to the input and help the model differentiate the order of the sequence: $f_{0,i}=e(i) + x_i$ where $x_i$ is the $i$th input, $f$ the input to your first layer and $e$ is a function the returns the $i$th positional embedding.

We'll focus on the learnt positional embeddings, which require us to learn $S$ different $D$ length embeddings which we will add to our input. The first question you may ask is what is $S$? This will be the *maximum* length of the sequence that will be inputted, you can work this out based on the image sizes as well as the patch sizes (don't forget about your CLS token too!)

**TASK 11.1:** Within the constructor for `CIFAR_Transformer`, create and randomly initialise the positional embeddings as an `nn.Parameter`.

Once these have been created, the next step is to add them to our input after it has been patchified. Similar to the CLS token, we will need to repeat this across the batch - think about why we *don't* want to learn a $B \times S \times D$ size set of positional embeddings.

**TASK 11.2:** In the forward pass, add the positional embeddings to the output of the patchify function following the VIT diagram above (remember we don't have the linear projectiion).

Afterward, run the model and check to see how the performance changes with the positional embeddings.

**Assessed Question:** In this lab, we have used learnt positional embeddings. What is the other common method for positional embeddings?

---
# Multihead Attention from scratch

For this extension, see the attention module we've added to `train_cifar.py`. Appendix A on page 13 in [An Image is Worth 16x16 Words](https://arxiv.org/pdf/2010.11929) may come in handy as a reference.

```python
class Attention(nn.Module):
    def __init__(self, dim=192, num_heads=1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=False)

        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, S, D = x.shape

        qkv = self.qkv(x).reshape(B, S, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        #TASK 12.1 Multiply by scale

        #TASK 12.2 Multiply queries by keys (transposed)

        #TASK 12.3 Apply softmax

        #TASK 12.4 Multiply the attention and values, then concatenate across heads

        x = self.proj(x)
        return x
```

**TASK 12.1:** First you have to multiply the queries $Q$ by $1/\sqrt{D/H}$, which we've precalculated as `self.scale`.

Take a careful note that $Q$, $K$ and $V$ are computed for you, and that they all have shape $(B, H, S, D/H)$, where $H$ is the number of heads.

**TASK 12.2:** Now you have to multiply the queries $Q$ by the transposed keys $K^T$, assigning this to the variable `attn`.

Remember that by transposing, we want to swap the dimensions $S$ and $D/H$, not the batch or sequence dimension.

**TASK 12.3:** Apply softmax to `attn`, normalising over the last dimension and setting this value again to `attn`.

You can use `F.softmax` as we've imported `torch.nn.functional` as `F`.

**TASK 12.4:** The final step is to multiply `attn` by our values $V$, and then concatenate the result across heads.

This step is a bit more awkward and the shapes of the tensors is very important. Keep in mind that the input to this forward pass is of shape $(B, S, D)$ *and* the output should also be the same shape. Notably, the `reshape` function can be used to concatenate across heads, by reshaping from a tensor of shape $(B, S, H, D/H)$ to $(B, S, D)$, which is what we want to output.

Now that you've implemented the forward pass for your attention module, you can try instantiating it somewhere, altering the hidden dimension and the number of heads to see how the shape of the intermediate variables changes.

___

# The State-of-the-Art??

This is the end of all the tasks within the lab, but there is much left to explore and play with for the transformer. You may be interested in modifying the patch size, adding in the linear projection layer, increasing the depth, or modifying the sizes of the feed-forward networks within each encoder layer. Of course there are a whole host of hyperparameters to tune. The lessons learnt over the past two labs can also be applied here, e.g. does data augmentation help the model?

With that said, this takes us to the end of Lab 5, and the last learnt lab for the unit. From humble beginnings of classifying irises with a 2 layer fully-connected network, we have now created a model for CIFAR-10 which is based on the *state-of-the-art*.